In [1]:
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor

def create_spatial_model(file_path='CGWB_data_.csv'):
    """
    Loads data, trains a spatial model (K-Nearest Neighbors), and returns it.
    """
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: {file_path} not found. Cannot create the spatial model.")
        return None

    # Melt the dataframe to long format
    id_vars = ['STATE', 'DISTRICT', 'LAT', 'LON', 'SITE_TYPE', 'WLCODE']
    df_long = pd.melt(df, id_vars=id_vars, var_name='Date', value_name='Water_Level')

    # Convert 'Date' and handle missing values
    df_long['Date'] = pd.to_datetime(df_long['Date'], format='%d/%m/%Y')
    df_long.dropna(subset=['Water_Level', 'LAT', 'LON'], inplace=True)

    # Find the most recent water level for each well
    df_recent = df_long.loc[df_long.groupby('WLCODE')['Date'].idxmax()]

    # Define features (X) and target (y)
    X = df_recent[['LAT', 'LON']]
    y = df_recent['Water_Level']

    # Initialize and train the K-Nearest Neighbors model
    # n_neighbors=5 means it will look at the 5 nearest wells.
    # weights='distance' gives more importance to closer wells.
    knn_model = KNeighborsRegressor(n_neighbors=5, weights='distance')
    knn_model.fit(X, y)

    print("Spatial prediction model trained successfully.")
    return knn_model

def predict_water_level(latitude, longitude, model):
    """
    Predicts the groundwater level for a given latitude and longitude using the trained model.
    """
    if model is None:
        return "Model not available. Please train the model first."

    # Create a DataFrame for the input
    input_data = pd.DataFrame([[latitude, longitude]], columns=['LAT', 'LON'])

    # Make the prediction
    prediction = model.predict(input_data)

    return prediction[0]

# --- Main Execution ---
# 1. Create and train the model
spatial_model = create_spatial_model('CGWB_data_cleaned.csv')

# 2. Use the model to make a prediction for a new location
if spatial_model:
    # Example: Predict for your location in Ghaziabad, Uttar Pradesh
    input_lat = input("enter the latitude")
    input_lon = input("enter the logitude")

    predicted_level = predict_water_level(input_lat, input_lon, spatial_model)

    print(f"\n--- Groundwater Level Prediction by Location ---")
    print(f"For Latitude: {input_lat}, Longitude: {input_lon} (Ghaziabad)")
    print(f"The predicted groundwater level is: {predicted_level:.2f} meters")

Spatial prediction model trained successfully.


enter the latitude 28.65
enter the logitude 78.03333333



--- Groundwater Level Prediction by Location ---
For Latitude: 28.65, Longitude: 78.03333333 (Ghaziabad)
The predicted groundwater level is: 10.04 meters
